In [1]:
# =============================
# IMPORT LIBRARIES
# =============================

import os
import cv2
import numpy as np
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split

from tensorflow.keras.utils import to_categorical
from tensorflow.keras.models import Sequential

from tensorflow.keras.layers import (
    Conv2D,
    MaxPooling2D,
    Dense,
    Flatten,
    Dropout
)

from tensorflow.keras.optimizers import Adam

# =============================
# LOAD DATASET
# =============================

data = []
labels = []

IMG_SIZE = 128

categories = [
    'glioma',
    'meningioma',
    'notumor',
    'pituitary'
]

dataset_path = "/kaggle/input/datasets/masoudnickparvar/brain-tumor-mri-dataset/Training"

for category in categories:

    path = os.path.join(dataset_path, category)

    label = categories.index(category)

    print("Loading:", category)

    for img in os.listdir(path):

        try:

            img_path = os.path.join(path, img)

            image = cv2.imread(img_path)

            image = cv2.resize(
                image,
                (IMG_SIZE, IMG_SIZE)
            )

            data.append(image)
            labels.append(label)

        except:
            pass

print("Dataset Loaded")

# =============================
# CONVERT TO NUMPY
# =============================

X = np.array(data) / 255.0
y = np.array(labels)

print("X Shape:", X.shape)
print("y Shape:", y.shape)

# =============================
# ONE HOT ENCODING
# =============================

y = to_categorical(y, num_classes=4)

print("Encoded y Shape:", y.shape)

# =============================
# TRAIN TEST SPLIT
# =============================

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

print("X_train:", X_train.shape)
print("X_test :", X_test.shape)

print("y_train:", y_train.shape)
print("y_test :", y_test.shape)

2026-05-12 10:13:32.295632: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1778580812.722734      57 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1778580812.858366      57 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1778580813.783765      57 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1778580813.783818      57 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1778580813.783821      57 computation_placer.cc:177] computation placer alr

Loading: glioma
Loading: meningioma
Loading: notumor
Loading: pituitary
Dataset Loaded
X Shape: (5600, 128, 128, 3)
y Shape: (5600,)
Encoded y Shape: (5600, 4)
X_train: (4480, 128, 128, 3)
X_test : (1120, 128, 128, 3)
y_train: (4480, 4)
y_test : (1120, 4)


In [2]:
# =============================
# BUILD CNN MODEL
# =============================

model = Sequential()

# First CNN Layer
model.add(
    Conv2D(
        32,
        (3,3),
        activation='relu',
        input_shape=(128,128,3)
    )
)

model.add(MaxPooling2D(2,2))

# Second CNN Layer
model.add(
    Conv2D(
        64,
        (3,3),
        activation='relu'
    )
)

model.add(MaxPooling2D(2,2))

# Third CNN Layer
model.add(
    Conv2D(
        128,
        (3,3),
        activation='relu'
    )
)

model.add(MaxPooling2D(2,2))

# Flatten Layer
model.add(Flatten())

# Dense Layer
model.add(Dense(128, activation='relu'))

# Dropout
model.add(Dropout(0.5))

# Output Layer
model.add(Dense(4, activation='softmax'))

# =============================
# COMPILE MODEL
# =============================

model.compile(
    optimizer=Adam(learning_rate=0.0001),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

# =============================
# MODEL SUMMARY
# =============================

model.summary()

/usr/local/lib/python3.12/dist-packages/keras/src/layers/convolutional/base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)
2026-05-12 10:14:57.569991: E external/local_xla/xla/stream_executor/cuda/cuda_platform.cc:51] failed call to cuInit: INTERNAL: CUDA error: Failed call to cuInit: UNKNOWN ERROR (303)


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv2d (Conv2D)                 │ (None, 126, 126, 32)   │           896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d (MaxPooling2D)    │ (None, 63, 63, 32)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_1 (Conv2D)               │ (None, 61, 61, 64)     │        18,496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_1 (MaxPooling2D)  │ (None, 30, 30, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_2 (Conv2D)               │ (None, 28, 28, 128)    │        73,856 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_2 (MaxPooling2D)  │ (None, 14, 14, 128)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten (Flatten)               │ (None, 25088)          │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 128)            │     3,211,392 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 4)              │           516 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 3,305,156 (12.61 MB)

 Trainable params: 3,305,156 (12.61 MB)

 Non-trainable params: 0 (0.00 B)

In [3]:
# =============================
# TRAIN MODEL
# =============================

history = model.fit(
    X_train,
    y_train,
    epochs=25,
    validation_data=(X_test, y_test),
    batch_size=32
)

Epoch 1/25
140/140 ━━━━━━━━━━━━━━━━━━━━ 82s 570ms/step - accuracy: 0.4852 - loss: 1.1569 - val_accuracy: 0.7054 - val_loss: 0.7426
Epoch 2/25
140/140 ━━━━━━━━━━━━━━━━━━━━ 76s 540ms/step - accuracy: 0.7180 - loss: 0.7254 - val_accuracy: 0.7643 - val_loss: 0.5757
Epoch 3/25
140/140 ━━━━━━━━━━━━━━━━━━━━ 71s 507ms/step - accuracy: 0.7877 - loss: 0.5845 - val_accuracy: 0.7902 - val_loss: 0.4917
Epoch 4/25
140/140 ━━━━━━━━━━━━━━━━━━━━ 71s 505ms/step - accuracy: 0.8129 - loss: 0.5099 - val_accuracy: 0.8402 - val_loss: 0.4329
Epoch 5/25
140/140 ━━━━━━━━━━━━━━━━━━━━ 73s 522ms/step - accuracy: 0.8292 - loss: 0.4630 - val_accuracy: 0.8446 - val_loss: 0.4054
Epoch 6/25
140/140 ━━━━━━━━━━━━━━━━━━━━ 68s 486ms/step - accuracy: 0.8457 - loss: 0.4161 - val_accuracy: 0.8518 - val_loss: 0.3889
Epoch 7/25
140/140 ━━━━━━━━━━━━━━━━━━━━ 74s 530ms/step - accuracy: 0.8595 - loss: 0.3829 - val_accuracy: 0.8607 - val_loss: 0.3755
Epoch 8/25
140/140 ━━━━━━━━━━━━━━━━━━━━ 77s 548ms/step - accuracy: 0.8751 - loss: 0

In [4]:
# =============================
# EVALUATE MODEL
# =============================

loss, accuracy = model.evaluate(
    X_test,
    y_test
)

print(f"\nAccuracy: {accuracy * 100:.2f}%")

35/35 ━━━━━━━━━━━━━━━━━━━━ 5s 132ms/step - accuracy: 0.9394 - loss: 0.1827

Accuracy: 92.95%


In [6]:
# =============================
# SAVE MODEL
# =============================

model.save("/kaggle/working/brain_tumor_model1.keras")

print("Model Saved Successfully")

Model Saved Successfully
